# Airbnb SBERT Host and Review Embeddings

Run this notebook in Google Colab with a GPU runtime. It builds Sentence-BERT embeddings for `host_about` and pre-cutoff guest review text, then writes mergeable feature files keyed by `city`, `snapshot`, and `listing_id`.

These embeddings are intended to supplement the local feature builder: base controls, hand-built host/review NLP features, and SBERT semantic embeddings.

In [ ]:
!pip -q install sentence-transformers pyarrow tqdm

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import html
import re

import pandas as pd
from sentence_transformers import SentenceTransformer

RAW_DIR = Path("/content/drive/MyDrive/comp90051/data/raw/inside_airbnb_australia")
OUT_DIR = Path("/content/drive/MyDrive/comp90051/data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 512
MAX_REVIEWS_PER_LISTING = 50
MAX_TEXT_CHARS = 8000

In [ ]:
TAG_RE = re.compile(r"<[^>]+>")
MOJIBAKE_REPLACEMENTS = {
    "‚Äô": "'",
    "‚Äò": "'",
    "‚Äú": '"',
    "‚Äù": '"',
    "‚Äì": "-",
    "‚Äî": "-",
    "¬†": " ",
}


def clean_text(value):
    if value is None or pd.isna(value):
        return ""
    text = html.unescape(str(value))
    for bad, good in MOJIBAKE_REPLACEMENTS.items():
        text = text.replace(bad, good)
    text = text.replace("\r", " ").replace("\n", " ")
    text = TAG_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def embed_texts(model, texts, prefix):
    vectors = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return pd.DataFrame(
        vectors, columns=[f"{prefix}_sbert_{i:03d}" for i in range(vectors.shape[1])]
    )

In [ ]:
manifest = pd.read_csv(RAW_DIR / "manifest.csv")
snapshots = manifest.pivot_table(
    index=["city", "snapshot"],
    columns="file_name",
    values="url",
    aggfunc="first",
).reset_index()
snapshots.head()

In [ ]:
model = SentenceTransformer(MODEL_NAME, device="cuda")

In [ ]:
parts = []
for row in snapshots.itertuples(index=False):
    city = row.city
    snapshot = row.snapshot
    listing_path = RAW_DIR / city / snapshot / "listings.csv.gz"
    review_path = RAW_DIR / city / snapshot / "reviews.csv.gz"
    if not listing_path.exists() or not review_path.exists():
        continue

    listings = pd.read_csv(listing_path, usecols=["id", "host_about"], low_memory=False)
    reviews = pd.read_csv(review_path, usecols=["listing_id", "date", "comments"], low_memory=False)
    reviews["date"] = pd.to_datetime(reviews["date"], errors="coerce")
    reviews = reviews[reviews["date"] < pd.Timestamp(snapshot)]
    reviews = reviews.sort_values(["listing_id", "date"], ascending=[True, False])
    reviews = reviews.groupby("listing_id", sort=False).head(MAX_REVIEWS_PER_LISTING)
    reviews["comments"] = reviews["comments"].map(clean_text)
    review_text = reviews.groupby("listing_id")["comments"].agg(" ".join).str[:MAX_TEXT_CHARS]

    frame = pd.DataFrame({"listing_id": listings["id"].astype("int64")})
    frame["city"] = city
    frame["snapshot"] = snapshot
    frame["host_about_text"] = listings["host_about"].map(clean_text).str[:MAX_TEXT_CHARS]
    frame = frame.merge(
        review_text.rename("review_text"), how="left", left_on="listing_id", right_index=True
    )
    frame["review_text"] = frame["review_text"].fillna("")

    print(f"Encoding {city} {snapshot}: {len(frame):,} listings")
    host_emb = embed_texts(model, frame["host_about_text"].tolist(), "host_about")
    review_emb = embed_texts(model, frame["review_text"].tolist(), "reviews")
    keys = frame[["city", "snapshot", "listing_id"]].reset_index(drop=True)
    features = pd.concat([keys, host_emb, review_emb], axis=1)
    features.to_parquet(OUT_DIR / f"airbnb_sbert_features_{city}_{snapshot}.parquet", index=False)
    parts.append(features)

sbert_features = pd.concat(parts, ignore_index=True)
sbert_features.to_parquet(OUT_DIR / "airbnb_sbert_features.parquet", index=False)
sbert_features.shape